# 📊 US Stock ML Feature Builder
**목적**: 주가 예측 분류 ML 모형용 피처 수집 및 가공  
**타겟**: t+3개월 시장 대비 초과수익률 5단계 분류  
**테스트**: 시가총액 상위 10개 종목  

| 섹션 | 피처 | 데이터 소스 |
|------|------|-------------|
| Cell 1 | 환경 설정 | - |
| Cell 2 | 종목 선정 (시총 상위 10) | DB (us_stock_daily_market_cap) |
| Cell 3 | Price / Technical Features | DB + yfinance |
| Cell 4 | Macro Features | FDR / FRED |
| Cell 5 | Commodity Features | FDR / yfinance |
| Cell 6 | Fundamental Features | FMP API |
| Cell 7 | Valuation Features (TTM) | FMP API |
| Cell 8 | Sector Features | FMP API |
| Cell 9 | Analyst Estimate Revision | FMP API |
| Cell 10 | 피처 통합 및 최종 결과 확인 | - |

---
## Cell 1 — 환경 설정 및 라이브러리 임포트

In [1]:
import sys, os, warnings, time
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import numpy as np
import pandas as pd
import pymysql
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── DATA 폴더 자동 감지 (노트북 / 데스크탑 공용) ─────────────────────
DATA_PATHS = [
    r'C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA',
    r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA',
]
DATA_DIR = next((p for p in DATA_PATHS if os.path.isdir(p)), None)
assert DATA_DIR, f'DATA 폴더를 찾을 수 없습니다.\n시도한 경로:\n' + '\n'.join(DATA_PATHS)
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)

from config import get_db_info
from us_target_ticker_list_2000 import ticker_list
print(f'✅ DATA 폴더: {DATA_DIR}')
print(f'✅ 티커 리스트: {len(ticker_list)}개')

# ── API 설정 ──────────────────────────────────────────────────────────
FMP_API_KEY  = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
FMP_BASE     = 'https://financialmodelingprep.com/api/v3'
FMP_BASE_V4  = 'https://financialmodelingprep.com/api/v4'
FRED_API_KEY = ''  # FRED API Key (https://fred.stlouisfed.org/docs/api/api_key.html) — 없으면 pandas_datareader 사용
RATE_SLEEP   = 0.26  # FMP Starter: ~250콜/분 → 0.26초 간격

# ── 기준일 설정 ────────────────────────────────────────────────────────
T0       = datetime.today().replace(day=1) - relativedelta(months=1)  # 전월 말 기준
T0_STR   = T0.strftime('%Y-%m-%d')
LOOKBACK = 365  # 피처 계산용 주가 look-back (일)
print(f'\n⚙️  기준일(T0): {T0_STR}')
print(f'⚙️  주가 look-back: {LOOKBACK}일')

# ── DB 연결 함수 ──────────────────────────────────────────────────────
def get_conn():
    d = get_db_info()
    return pymysql.connect(
        host=d['host'], port=int(d['port']),
        user=d['user'], password=d['password'],
        database=d['database'], charset='utf8mb4', autocommit=False
    )

# ── FMP GET 헬퍼 ─────────────────────────────────────────────────────
def fmp_get(endpoint: str, params: dict = None, v4: bool = False) -> list:
    base = FMP_BASE_V4 if v4 else FMP_BASE
    url  = f'{base}/{endpoint}'
    p    = {'apikey': FMP_API_KEY}
    if params:
        p.update(params)
    try:
        r = requests.get(url, params=p, timeout=15)
        r.raise_for_status()
        data = r.json()
        return data if isinstance(data, list) else [data]
    except Exception as e:
        print(f'  ⚠️  FMP 오류 [{endpoint}]: {e}')
        return []
    finally:
        time.sleep(RATE_SLEEP)

print('\n✅ 설정 완료')
# ── Series 정규화 공통 헬퍼 (DatetimeIndex + tz-naive 보장) ─────────
def normalize_series(s) -> pd.Series:
    """
    DataFrame / Series 어느 형태든 pd.Series(DatetimeIndex, tz-naive)로 변환.
    RangeIndex / 변환 불가 index → 빈 Series 반환.
    """
    if s is None:
        return pd.Series(dtype=float)
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0] if not s.empty else pd.Series(dtype=float)
    if not isinstance(s, pd.Series) or s.empty:
        return pd.Series(dtype=float)
    if not isinstance(s.index, pd.DatetimeIndex):
        try:
            s = s.copy()
            s.index = pd.to_datetime(s.index)
        except Exception:
            return pd.Series(dtype=float)
    if s.index.tz is not None:
        s = s.copy()
        s.index = s.index.tz_localize(None)
    return pd.to_numeric(s, errors='coerce').dropna().sort_index()

print('✅ normalize_series 헬퍼 등록 완료')


✅ DATA 폴더: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✅ 티커 리스트: 2000개

⚙️  기준일(T0): 2026-03-01
⚙️  주가 look-back: 365일

✅ 설정 완료
✅ normalize_series 헬퍼 등록 완료


---
## Cell 2 — 시가총액 상위 10개 종목 선정 (DB 조회)

In [2]:
# ── DB에서 최근 시가총액 기준 상위 10개 종목 조회 ─────────────────────
N_TEST = 10  # ← 테스트 종목 수 (전체 실행 시 len(ticker_list)로 변경)

conn = get_conn()
cur  = conn.cursor()

# 최신 날짜의 market_cap 기준 상위 N개 종목
# indicator = 'market_cap' 이 없으면 close_price 최신 날짜 기준 종목 수집
sql = """
    SELECT t.ticker, t.value AS market_cap, t.date
    FROM us_stock_daily_market_cap t
    INNER JOIN (
        SELECT ticker, MAX(date) AS max_date
        FROM us_stock_daily_market_cap
        WHERE indicator = 'market_cap'
        GROUP BY ticker
    ) m ON t.ticker = m.ticker AND t.date = m.max_date
    WHERE t.indicator = 'market_cap'
    ORDER BY t.value DESC
    LIMIT %s
"""
cur.execute(sql, (N_TEST,))
rows = cur.fetchall()
conn.close()

if rows:
    df_top = pd.DataFrame(rows, columns=['ticker', 'market_cap', 'date'])
    TEST_TICKERS = df_top['ticker'].tolist()
    print(f'✅ DB에서 시총 상위 {N_TEST}개 종목 선정')
    display(df_top)
else:
    # DB에 market_cap이 없으면 하드코딩 fallback
    print('⚠️  DB에서 market_cap 데이터 없음 → 시총 상위 10개 하드코딩 사용')
    TEST_TICKERS = ['AAPL','MSFT','NVDA','AMZN','GOOGL','META','BRK-B','LLY','AVGO','TSLA']
    df_top = pd.DataFrame({'ticker': TEST_TICKERS})
    display(df_top)

print(f'\n🎯 테스트 종목: {TEST_TICKERS}')

✅ DB에서 시총 상위 10개 종목 선정


,ticker,market_cap,date
0,NVDA,4311286559999.0000,2026-04-02
1,AAPL,3774348595360.0000,2026-04-02
2,GOOG,3570831210000.0000,2026-04-02
3,MSFT,2775181260000.0000,2026-04-02
4,AMZN,2246426930000.0000,2026-04-02
5,TSM,1758583392384.0000,2026-04-02
6,AVGO,1491281550000.0000,2026-04-02
7,META,1448213660000.0000,2026-04-02
8,TSLA,1165066290000.0000,2026-04-02
9,WMT,1002672090000.0000,2026-04-02



🎯 테스트 종목: ['NVDA', 'AAPL', 'GOOG', 'MSFT', 'AMZN', 'TSM', 'AVGO', 'META', 'TSLA', 'WMT']


---
## Cell 3 — Price / Technical Features
> **소스**: DB (`us_stock_daily_market_cap` / `close_price`) + yfinance (거래량)

In [3]:
import yfinance as yf

# ── DB에서 close_price 추출 ────────────────────────────────────────────
def fetch_close_from_db(tickers: list, lookback_days: int = 400) -> pd.DataFrame:
    """DB us_stock_daily_market_cap 에서 close_price 추출 → wide format 반환"""
    from_date = (datetime.today() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    conn = get_conn()
    cur  = conn.cursor()
    placeholders = ','.join(['%s'] * len(tickers))
    sql = f"""
        SELECT date, ticker, value
        FROM us_stock_daily_market_cap
        WHERE indicator = 'close_price'
          AND ticker IN ({placeholders})
          AND date >= %s
        ORDER BY date
    """
    cur.execute(sql, tickers + [from_date])
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows, columns=['date', 'ticker', 'close'])
    df['date']  = pd.to_datetime(df['date'])
    df['close'] = df['close'].astype(float)
    pivot = df.pivot(index='date', columns='ticker', values='close').sort_index()
    # tz-naive 보장
    if pivot.index.tz is not None:
        pivot.index = pivot.index.tz_localize(None)
    return pivot

# ── yfinance에서 OHLCV 수집 ──────────────────────────────────────────
def fetch_ohlcv_yf(tickers: list, lookback_days: int = 400) -> dict:
    """yfinance OHLCV → {ticker: DataFrame(columns=소문자)} 반환"""
    from_date = (datetime.today() - timedelta(days=lookback_days)).strftime('%Y-%m-%d')
    result = {}
    for tk in tickers:
        try:
            raw = yf.download(tk, start=from_date, progress=False, auto_adjust=True)
            if raw.empty:
                continue
            # MultiIndex 컬럼 처리 (yfinance ≥0.2 단일 티커도 MultiIndex 반환 가능)
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = [c[0].lower() for c in raw.columns]
            else:
                raw.columns = [str(c).lower() for c in raw.columns]
            # tz-naive 보장
            if raw.index.tz is not None:
                raw.index = raw.index.tz_localize(None)
            result[tk] = raw
        except Exception as e:
            print(f'  ⚠️  yfinance [{tk}]: {e}')
    return result

# ── 피처 계산 ────────────────────────────────────────────────────────
def calc_price_features(ticker: str, price_db: pd.DataFrame, ohlcv_yf: dict, t0) -> dict:
    feat = {'ticker': ticker}
    cutoff = pd.Timestamp(t0)

    # close series: DB 우선, 없으면 yfinance
    if not price_db.empty and ticker in price_db.columns and price_db[ticker].dropna().shape[0] > 20:
        raw_s = price_db[ticker].dropna()
    elif ticker in ohlcv_yf and not ohlcv_yf[ticker].empty and 'close' in ohlcv_yf[ticker].columns:
        raw_s = ohlcv_yf[ticker]['close'].dropna()
    else:
        return feat

    s = raw_s[raw_s.index <= cutoff].copy()
    if len(s) < 20:
        return feat

    p0 = float(s.iloc[-1])

    # ── 모멘텀 ──────────────────────────────────────────────────────
    def ret_n(n):
        if len(s) <= n: return float('nan')
        return (p0 / float(s.iloc[-n]) - 1) * 100

    feat['ret_1m']  = ret_n(21)
    feat['ret_3m']  = ret_n(63)
    feat['ret_6m']  = ret_n(126)
    feat['ret_12m'] = ret_n(252)
    feat['ret_12m_ex_1m'] = (float(s.iloc[-21]) / float(s.iloc[-252]) - 1) * 100 if len(s) > 252 else float('nan')

    # ── 이동평균 ────────────────────────────────────────────────────
    for w in [20, 60, 120]:
        feat[f'price_to_ma{w}'] = (p0 / float(s.iloc[-w:].mean()) - 1) * 100 if len(s) >= w else float('nan')
    if len(s) >= 60:
        feat['ma20_to_ma60']   = (float(s.iloc[-20:].mean()) / float(s.iloc[-60:].mean()) - 1) * 100
    if len(s) >= 120:
        feat['ma60_to_ma120']  = (float(s.iloc[-60:].mean()) / float(s.iloc[-120:].mean()) - 1) * 100

    # ── 변동성 ──────────────────────────────────────────────────────
    log_ret = np.log(s / s.shift(1)).dropna()
    if len(log_ret) >= 20:
        feat['vol_20d'] = float(log_ret.iloc[-20:].std()) * np.sqrt(252) * 100
    if len(log_ret) >= 60:
        lr60 = log_ret.iloc[-60:]
        feat['vol_60d']          = float(lr60.std()) * np.sqrt(252) * 100
        neg = lr60[lr60 < 0]
        feat['downside_vol_60d'] = float(neg.std()) * np.sqrt(252) * 100 if len(neg) > 1 else float('nan')
        v20 = float(log_ret.iloc[-20:].std())
        v60 = float(lr60.std())
        feat['vol_change_20d_60d'] = (v20 / v60 - 1) * 100 if v60 > 0 else float('nan')

    # ── 52주 고저 ───────────────────────────────────────────────────
    if len(s) >= 252:
        h252 = float(s.iloc[-252:].max()); l252 = float(s.iloc[-252:].min())
        feat['drawdown_252d']      = (p0 / h252 - 1) * 100
        feat['price_to_high_252d'] = (p0 / h252 - 1) * 100
        feat['price_to_low_252d']  = (p0 / l252 - 1) * 100

    # ── 거래량 (yfinance) ────────────────────────────────────────────
    if ticker in ohlcv_yf:
        yf_df = ohlcv_yf[ticker]
        yf_df = yf_df[yf_df.index <= cutoff]
        if 'volume' in yf_df.columns and len(yf_df) >= 20:
            vol = yf_df['volume'].dropna().astype(float)
            v_mean = float(vol.iloc[-20:].mean())
            v_std  = float(vol.iloc[-20:].std())
            feat['volume_zscore_20d'] = (float(vol.iloc[-1]) - v_mean) / v_std if v_std > 0 else float('nan')
            if len(vol) >= 42:
                feat['volume_change_1m'] = (float(vol.iloc[-21:].mean()) / float(vol.iloc[-42:-21].mean()) - 1) * 100
            if 'close' in yf_df.columns and len(vol) >= 60:
                cl_yf = yf_df['close'].dropna().astype(float)
                dv = (cl_yf * vol.reindex(cl_yf.index)).dropna()
                feat['dollar_volume_60d'] = float(dv.iloc[-60:].mean()) if len(dv) >= 60 else float('nan')
            # Amihud
            if len(log_ret) >= 60 and len(vol) >= 60:
                lr_abs = log_ret.iloc[-60:].abs()
                dv_am  = (s.iloc[-60:] * vol.reindex(s.index).iloc[-60:]).replace(0, float('nan'))
                feat['amihud_illiquidity'] = float((lr_abs / dv_am).mean()) * 1e6
        # RSI 14
        if 'close' in yf_df.columns and len(yf_df) >= 15:
            delta = yf_df['close'].astype(float).diff().dropna()
            gain  = float(delta.clip(lower=0).iloc[-14:].mean())
            loss  = float((-delta.clip(upper=0)).iloc[-14:].mean())
            feat['rsi_14d'] = 100 - 100 / (1 + gain / loss) if loss > 0 else 100.0

    return feat

# ── 실행 ──────────────────────────────────────────────────────────────
print('📥 DB close_price 수집 중...')
price_db = fetch_close_from_db(TEST_TICKERS, lookback_days=LOOKBACK + 50)
n_db = len(price_db.columns) if not price_db.empty else 0
print(f'  → DB 종목: {n_db}개  행: {len(price_db)}')

print('📥 yfinance OHLCV 수집 중...')
ohlcv_yf = fetch_ohlcv_yf(TEST_TICKERS, lookback_days=LOOKBACK + 50)
print(f'  → yfinance 수집: {len(ohlcv_yf)}개 종목')

print('\n⚙️  Price/Technical 피처 계산 중...')
price_features = []
for tk in TEST_TICKERS:
    f = calc_price_features(tk, price_db, ohlcv_yf, T0)
    n_feat = sum(1 for k, v in f.items() if k != 'ticker' and not (isinstance(v, float) and np.isnan(v)))
    print(f'  ✅ {tk}: {n_feat}개 유효 피처')
    price_features.append(f)

df_price = pd.DataFrame(price_features).set_index('ticker')
print(f'\n✅ Price/Technical 피처 완성: {df_price.shape[1]}개 컬럼')
display(df_price.round(4))


📥 DB close_price 수집 중...
  → DB 종목: 10개  행: 283
📥 yfinance OHLCV 수집 중...
  → yfinance 수집: 10개 종목

⚙️  Price/Technical 피처 계산 중...
  ✅ NVDA: 22개 유효 피처
  ✅ AAPL: 22개 유효 피처
  ✅ GOOG: 22개 유효 피처
  ✅ MSFT: 22개 유효 피처
  ✅ AMZN: 22개 유효 피처
  ✅ TSM: 22개 유효 피처
  ✅ AVGO: 22개 유효 피처
  ✅ META: 22개 유효 피처
  ✅ TSLA: 22개 유효 피처
  ✅ WMT: 22개 유효 피처

✅ Price/Technical 피처 완성: 22개 컬럼


,ret_1m,ret_3m,ret_6m,ret_12m,ret_12m_ex_1m,price_to_ma20,price_to_ma60,price_to_ma120,ma20_to_ma60,ma60_to_ma120,vol_20d,vol_60d,downside_vol_60d,vol_change_20d_60d,drawdown_252d,price_to_high_252d,price_to_low_252d,volume_zscore_20d,volume_change_1m,dollar_volume_60d,amihud_illiquidity,rsi_14d
ticker,,,,,,,,,,,,,,,,,,,,,,
NVDA,-7.9632,-1.7032,-1.6486,47.5025,60.2647,-4.7278,-4.1770,-4.1646,0.5781,0.0130,45.7928,34.8305,24.0967,31.4732,-14.4182,-14.4182,87.9096,2.0214,24.9463,32218073144.3352,0.0000,41.0962
AAPL,2.2843,-4.8172,13.7089,11.7040,9.2093,-1.6520,-1.3118,0.3840,0.3459,1.7183,34.0466,23.4063,17.7798,45.4591,-7.6907,-7.6907,53.7360,0.9411,20.4672,12925113926.2728,0.0000,38.9249
GOOG,-8.0405,-2.6994,46.8663,83.6803,99.7405,-2.0828,-2.6260,6.4053,-0.5547,9.2748,24.4446,22.0610,14.3087,10.8043,-9.7043,-9.7043,113.0456,0.8656,32.4377,7046832346.1526,0.0000,38.7723
MSFT,-9.4025,-19.1061,-22.7938,0.5891,11.0286,-2.5079,-13.0007,-18.1821,-10.7627,-5.9556,32.4255,32.9748,32.1187,-1.6660,-27.4129,-27.4129,11.3619,0.7545,83.8980,14366159918.8136,0.0000,44.8238
AMZN,-13.1262,-8.3610,-9.3264,0.6036,15.8043,-1.6009,-7.4930,-7.7723,-5.9880,-0.3019,33.8662,27.6748,20.1425,22.3720,-17.3228,-17.3228,25.5080,-0.2508,66.6490,10633168418.9606,0.0000,49.5774
TSM,10.3166,29.1833,57.2082,106.8474,87.5034,4.1762,14.2442,22.2206,9.6645,6.9819,38.3598,34.0232,20.7867,12.7460,-3.3915,-3.3915,164.9643,-0.9718,-11.3258,4184423456.5480,0.0000,65.9953
AVGO,-3.3804,-19.4703,3.9086,63.0108,68.7140,-2.8691,-7.0694,-8.0731,-4.3244,-1.0800,40.7826,44.4622,40.1787,-8.2758,-22.4732,-22.4732,119.7428,0.3330,0.0620,10537764064.0666,0.0000,40.6281
META,-12.2076,2.3835,-13.5748,-1.2177,12.5181,-2.0866,-1.2172,-4.1653,0.8879,-2.9844,27.1437,33.7343,13.7968,-19.5369,-17.8292,-17.8292,34.0433,0.5041,10.4601,10370119014.7049,0.0000,44.3277
TSLA,-3.3729,-5.6426,16.3391,42.7594,47.7425,-2.7441,-8.2081,-6.9665,-5.6182,1.3526,33.9913,36.7285,22.6189,-7.4525,-17.8350,-17.8350,81.4252,-0.3848,0.1958,29233072178.0467,0.0000,44.7031


---
## Cell 4 — Macro Features
> **소스**: FDR (FinanceDataReader) / pandas_datareader (FRED) / yfinance

In [4]:
import FinanceDataReader as fdr
try:
    import pandas_datareader as pdr
    HAS_PDR = True
except ImportError:
    HAS_PDR = False
    print('⚠️  pandas_datareader 미설치 → pip install pandas-datareader')

MACRO_FROM      = (T0 - relativedelta(months=3)).strftime('%Y-%m-%d')
MACRO_FROM_LONG = (T0 - relativedelta(months=14)).strftime('%Y-%m-%d')

# ── 공통 헬퍼: Series를 항상 DatetimeIndex + tz-naive로 정규화 ─────────
def normalize_series(s) -> pd.Series:
    """
    반환값이 DataFrame / Series 어느 형태든 안전하게 pd.Series(DatetimeIndex) 로 변환.
    - tz-aware index → tz-naive
    - RangeIndex → 변환 불가 → 빈 Series 반환
    """
    if s is None:
        return pd.Series(dtype=float)
    if isinstance(s, pd.DataFrame):
        if s.empty:
            return pd.Series(dtype=float)
        s = s.iloc[:, 0]
    if not isinstance(s, pd.Series):
        return pd.Series(dtype=float)
    if s.empty:
        return pd.Series(dtype=float)
    # RangeIndex 방어
    if not isinstance(s.index, pd.DatetimeIndex):
        try:
            s.index = pd.to_datetime(s.index)
        except Exception:
            return pd.Series(dtype=float)
    # tz-aware → tz-naive
    if s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    s = pd.to_numeric(s, errors='coerce').dropna()
    return s.sort_index()

def safe_fdr(symbol, start, end=None):
    try:
        end = end or T0.strftime('%Y-%m-%d')
        df = fdr.DataReader(symbol, start, end)
        return normalize_series(df)
    except Exception as e:
        print(f'  ⚠️  FDR [{symbol}]: {e}')
        return pd.Series(dtype=float)

def safe_fred(series_id, start):
    """FRED 수집 — pdr 우선, 실패 시 REST API fallback. 항상 정규화된 Series 반환."""
    # 방법1: pandas_datareader
    if HAS_PDR:
        try:
            df = pdr.DataReader(series_id, 'fred', start, T0)
            return normalize_series(df[series_id] if series_id in df.columns else df.iloc[:, 0])
        except Exception:
            pass
    # 방법2: FRED REST API
    try:
        url = 'https://api.stlouisfed.org/fred/series/observations'
        params = {
            'series_id': series_id,
            'observation_start': start,
            'observation_end': T0.strftime('%Y-%m-%d'),
            'api_key': FRED_API_KEY or 'none',
            'file_type': 'json',
        }
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        if 'observations' in data:
            tmp = pd.DataFrame(data['observations'])
            tmp['date']  = pd.to_datetime(tmp['date'])
            tmp['value'] = pd.to_numeric(tmp['value'], errors='coerce')
            s = tmp.set_index('date')['value']
            return normalize_series(s)
    except Exception as e:
        print(f'  ⚠️  FRED REST [{series_id}]: {e}')
    return pd.Series(dtype=float)

def latest_val(s: pd.Series, t0) -> float:
    """t0 이전 최신값 반환. 빈 Series 또는 잘못된 index도 안전하게 처리."""
    s = normalize_series(s)
    if s.empty:
        return float('nan')
    cutoff = pd.Timestamp(t0)
    s = s[s.index <= cutoff]
    return float(s.iloc[-1]) if len(s) > 0 else float('nan')

def change_1m(s: pd.Series, t0) -> float:
    """t0 기준 최근 두 값의 차이(월 변화량). 빈 Series 안전 처리."""
    s = normalize_series(s)
    if s.empty:
        return float('nan')
    cutoff = pd.Timestamp(t0)
    s = s[s.index <= cutoff]
    if len(s) < 2:
        return float('nan')
    return float(s.iloc[-1] - s.iloc[-2])

# ─────────────────────────────────────────────────────────────────────
print('📥 Macro 데이터 수집 중...')
macro_raw = {}

print('  금리 데이터 (FRED)...')
macro_raw['us10y']   = safe_fred('DGS10',          MACRO_FROM_LONG)
macro_raw['us2y']    = safe_fred('DGS2',            MACRO_FROM_LONG)
macro_raw['fedfund'] = safe_fred('FEDFUNDS',        MACRO_FROM_LONG)
macro_raw['realrate']= safe_fred('DFII10',          MACRO_FROM_LONG)  # 10y TIPS

print('  VIX...')
vix_raw = safe_fdr('VIX', MACRO_FROM_LONG)
if vix_raw.empty:
    vix_df = yf.download('^VIX', start=MACRO_FROM_LONG, progress=False, auto_adjust=True)
    vix_raw = normalize_series(vix_df['Close'] if not vix_df.empty else pd.Series(dtype=float))
macro_raw['vix'] = vix_raw

print('  DXY (yfinance)...')
dxy_df = yf.download('DX-Y.NYB', start=MACRO_FROM_LONG, progress=False, auto_adjust=True)
macro_raw['dxy'] = normalize_series(dxy_df['Close'] if not dxy_df.empty else pd.Series(dtype=float))

print('  신용 스프레드 (FRED HY OAS)...')
macro_raw['oas'] = safe_fred('BAMLH0A0HYM2', MACRO_FROM_LONG)

print('  PMI (FRED ISM)...')
macro_raw['pmi'] = safe_fred('NAPM', MACRO_FROM_LONG)

print('  실업률 (FRED)...')
macro_raw['unemployment'] = safe_fred('UNRATE', MACRO_FROM_LONG)

# ── 진단: 수집 현황 출력 ─────────────────────────────────────────────
print('\n📋 수집 결과:')
for k, v in macro_raw.items():
    n = len(v)
    last = str(v.index[-1].date()) if n > 0 else 'N/A'
    print(f'  {k:15s}: {n:3d}건  최신={last}')

# ── 피처 집계 ────────────────────────────────────────────────────────
print('\n⚙️  Macro 피처 계산 중...')
m = {}

m['us10y_yield']           = latest_val(macro_raw['us10y'],  T0)
m['us2y_yield']            = latest_val(macro_raw['us2y'],   T0)
ten  = m['us10y_yield'] if not np.isnan(m['us10y_yield']) else 0.0
two  = m['us2y_yield']  if not np.isnan(m['us2y_yield'])  else 0.0
m['term_spread_10y_2y']    = ten - two

# term_spread 변화율: 두 시리즈 공통 index로 맞춘 후 차이
us10 = macro_raw['us10y']
us2  = macro_raw['us2y'].reindex(us10.index, method='ffill')
spread_s = (us10 - us2).dropna()
m['term_spread_change_1m'] = change_1m(spread_s, T0)

m['real_rate_proxy']       = latest_val(macro_raw['realrate'],    T0)
m['vix']                   = latest_val(macro_raw['vix'],          T0)
m['vix_change_1m']         = change_1m(macro_raw['vix'],           T0)
m['oas_spread']            = latest_val(macro_raw['oas'],          T0)
m['credit_spread_change_1m']= change_1m(macro_raw['oas'],         T0)
m['dxy']                   = latest_val(macro_raw['dxy'],          T0)
m['dxy_change_1m']         = change_1m(macro_raw['dxy'],           T0)
m['pmi']                   = latest_val(macro_raw['pmi'],          T0)
m['pmi_change_1m']         = change_1m(macro_raw['pmi'],           T0)
m['unemployment_rate']     = latest_val(macro_raw['unemployment'], T0)
m['earnings_yield_spread'] = float('nan')   # Cell 7에서 업데이트

df_macro = pd.DataFrame([m])
print(f'\n✅ Macro 피처 완성: {df_macro.shape[1]}개')
display(df_macro.round(4))

macro_dict = m.copy()
print('\n(Macro 피처는 전 종목 공통값으로 적용됩니다)')


⚠️  pandas_datareader 미설치 → pip install pandas-datareader
📥 Macro 데이터 수집 중...
  금리 데이터 (FRED)...
  VIX...
  DXY (yfinance)...
  신용 스프레드 (FRED HY OAS)...
  PMI (FRED ISM)...
  실업률 (FRED)...

📋 수집 결과:
  us10y          :   0건  최신=N/A
  us2y           :   0건  최신=N/A
  fedfund        :   0건  최신=N/A
  realrate       :   0건  최신=N/A
  vix            : 290건  최신=2026-02-27
  dxy            : 316건  최신=2026-04-06
  oas            :   0건  최신=N/A
  pmi            :   0건  최신=N/A
  unemployment   :   0건  최신=N/A

⚙️  Macro 피처 계산 중...

✅ Macro 피처 완성: 15개


,us10y_yield,us2y_yield,term_spread_10y_2y,term_spread_change_1m,real_rate_proxy,vix,vix_change_1m,oas_spread,credit_spread_change_1m,dxy,dxy_change_1m,pmi,pmi_change_1m,unemployment_rate,earnings_yield_spread
0,NaN,NaN,0.0000,NaN,NaN,19.2800,1.2100,NaN,NaN,97.6100,-0.1800,NaN,NaN,NaN,NaN



(Macro 피처는 전 종목 공통값으로 적용됩니다)


---
## Cell 5 — Commodity Features
> **소스**: FDR / yfinance

In [5]:
COMM_FROM = (T0 - relativedelta(months=6)).strftime('%Y-%m-%d')

def fetch_commodity(symbol_fdr: str, symbol_yf: str, name: str) -> pd.Series:
    """FDR 우선, 실패 시 yfinance fallback. 항상 normalize_series 적용."""
    if symbol_fdr:
        s = safe_fdr(symbol_fdr, COMM_FROM)
        if not s.empty:
            return s
    try:
        df = yf.download(symbol_yf, start=COMM_FROM, progress=False, auto_adjust=True)
        if not df.empty:
            return normalize_series(df['Close'])
    except Exception as e:
        print(f'  ⚠️  yfinance [{name}/{symbol_yf}]: {e}')
    return pd.Series(dtype=float)

def comm_ret(s: pd.Series, t0, n_days: int) -> float:
    s = normalize_series(s)
    s = s[s.index <= pd.Timestamp(t0)]
    if len(s) <= n_days:
        return float('nan')
    return (float(s.iloc[-1]) / float(s.iloc[-n_days]) - 1) * 100

print('📥 원자재 가격 수집 중...')
comm_raw = {
    'wti':    fetch_commodity('CL=F', 'CL=F',  'WTI'),
    'copper': fetch_commodity('HG=F', 'HG=F',  'Copper'),
    'gold':   fetch_commodity('GC=F', 'GC=F',  'Gold'),
    'natgas': fetch_commodity('NG=F', 'NG=F',  'NatGas'),
    'corn':   fetch_commodity('ZC=F', 'ZC=F',  'Corn'),
}

print('\n📋 수집 결과:')
for k, v in comm_raw.items():
    n = len(v)
    last = str(v.index[-1].date()) if n > 0 else 'N/A'
    print(f'  {k:8s}: {n:3d}건  최신={last}')

print('\n⚙️  원자재 피처 계산 중...')
c = {}
for name, s in comm_raw.items():
    c[f'{name}_ret_1m'] = comm_ret(s, T0, 21)
    c[f'{name}_ret_3m'] = comm_ret(s, T0, 63)

# WTI 선물 커브 구조 (근월 vs 원월 근사)
try:
    cl1 = normalize_series(yf.download('CL=F',       start=COMM_FROM, progress=False, auto_adjust=True)['Close'])
    cl2 = normalize_series(yf.download('CLM26.NYM',  start=COMM_FROM, progress=False, auto_adjust=True)['Close'])
    if not cl1.empty and not cl2.empty:
        v1 = float(cl1.iloc[-1]); v2 = float(cl2.iloc[-1])
        c['wti_curve_structure'] = (v2 / v1 - 1) * 100 if v1 != 0 else float('nan')
    else:
        c['wti_curve_structure'] = float('nan')
except:
    c['wti_curve_structure'] = float('nan')

df_comm = pd.DataFrame([c])
print(f'\n✅ Commodity 피처 완성: {df_comm.shape[1]}개')
display(df_comm.round(4))

comm_dict = c.copy()


📥 원자재 가격 수집 중...

📋 수집 결과:
  wti     : 124건  최신=2026-02-27
  copper  : 124건  최신=2026-02-27
  gold    : 124건  최신=2026-02-27
  natgas  : 124건  최신=2026-02-27
  corn    : 124건  최신=2026-02-27

⚙️  원자재 피처 계산 중...

✅ Commodity 피처 완성: 11개


,wti_ret_1m,wti_ret_3m,copper_ret_1m,copper_ret_3m,gold_ret_1m,gold_ret_3m,natgas_ret_1m,natgas_ret_3m,corn_ret_1m,corn_ret_3m,wti_curve_structure
0,2.9134,12.5754,-3.2077,18.7033,-4.2284,25.6285,-24.3185,-36.8280,0.6395,2.3050,-11.8487


---
## Cell 6 — Fundamental Features
> **소스**: FMP API (Income Statement, Balance Sheet, Cash Flow / Quarterly TTM)

In [6]:
def fetch_fmp_financials(ticker: str) -> dict:
    """
    FMP에서 분기별 재무제표 3종 수집 후 Fundamental 피처 계산
    - Income Statement (quarterly)
    - Balance Sheet (quarterly)
    - Cash Flow (quarterly)
    데이터 없으면 빈 dict 반환
    """
    feat = {'ticker': ticker}

    inc  = fmp_get(f'income-statement/{ticker}',  {'period': 'quarter', 'limit': 8})
    bal  = fmp_get(f'balance-sheet-statement/{ticker}', {'period': 'quarter', 'limit': 8})
    cf   = fmp_get(f'cash-flow-statement/{ticker}', {'period': 'quarter', 'limit': 8})

    if not inc or not bal or not cf:
        print(f'  ⚠️  {ticker}: 재무제표 수집 실패 → 스킵')
        return feat

    def to_df(data):
        df = pd.DataFrame(data)
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date', ascending=False)
        return df

    inc_df = to_df(inc)
    bal_df = to_df(bal)
    cf_df  = to_df(cf)

    def safe_get(df, col, idx=0):
        try:
            v = df[col].iloc[idx]
            return float(v) if pd.notna(v) and v != 0 else np.nan
        except:
            return np.nan

    def ttm(df, col):
        """최근 4분기 합산 (TTM)"""
        try:
            return float(df[col].iloc[:4].sum())
        except:
            return np.nan

    # ── TTM 주요 항목 ────────────────────────────────────────────────
    rev_ttm   = ttm(inc_df, 'revenue')
    ni_ttm    = ttm(inc_df, 'netIncome')
    ebitda_ttm= ttm(inc_df, 'ebitda')
    gp_ttm    = ttm(inc_df, 'grossProfit')
    op_ttm    = ttm(inc_df, 'operatingIncome')
    cfo_ttm   = ttm(cf_df,  'operatingCashFlow')
    capex_ttm = ttm(cf_df,  'capitalExpenditure')

    # YoY 성장률 (최근 4분기 vs 직전 4분기)
    def yoy_growth(df, col):
        try:
            cur  = df[col].iloc[:4].sum()
            prev = df[col].iloc[4:8].sum()
            return (cur / abs(prev) - 1) * 100 if prev != 0 and pd.notna(prev) else np.nan
        except:
            return np.nan

    feat['sales_growth_yoy']   = yoy_growth(inc_df, 'revenue')
    feat['eps_growth_yoy']     = yoy_growth(inc_df, 'eps')
    feat['ebitda_growth_yoy']  = yoy_growth(inc_df, 'ebitda')
    feat['fcf_growth_yoy']     = yoy_growth(cf_df, 'freeCashFlow') if 'freeCashFlow' in cf_df.columns else np.nan

    # 수익성
    total_assets = safe_get(bal_df, 'totalAssets')
    total_equity = safe_get(bal_df, 'totalStockholdersEquity')
    feat['roe']              = (ni_ttm / total_equity * 100) if total_equity and ni_ttm else np.nan
    feat['roa']              = (ni_ttm / total_assets * 100) if total_assets and ni_ttm else np.nan
    feat['gross_margin']     = (gp_ttm / rev_ttm * 100) if rev_ttm and gp_ttm else np.nan
    feat['operating_margin'] = (op_ttm / rev_ttm * 100) if rev_ttm and op_ttm else np.nan

    # 재무 안정성
    total_debt = safe_get(bal_df, 'totalDebt')
    cash       = safe_get(bal_df, 'cashAndCashEquivalents')
    net_debt   = (total_debt - cash) if (total_debt and cash) else np.nan
    int_exp    = ttm(inc_df, 'interestExpense')
    feat['debt_to_equity']       = (total_debt / total_equity) if total_debt and total_equity else np.nan
    feat['net_debt_to_ebitda']   = (net_debt / ebitda_ttm) if (net_debt and ebitda_ttm) else np.nan
    feat['interest_coverage']    = (ebitda_ttm / abs(int_exp)) if (ebitda_ttm and int_exp and int_exp != 0) else np.nan

    # 효율성
    inventory   = safe_get(bal_df, 'inventory')
    cogs_ttm    = ttm(inc_df, 'costOfRevenue')
    feat['asset_turnover']     = (rev_ttm / total_assets) if rev_ttm and total_assets else np.nan
    feat['inventory_turnover'] = (cogs_ttm / inventory) if cogs_ttm and inventory else np.nan

    # 이익의 질 (Accruals)
    feat['cfo_to_net_income']  = (cfo_ttm / ni_ttm) if cfo_ttm and ni_ttm else np.nan
    feat['accruals_ratio']     = ((ni_ttm - cfo_ttm) / total_assets) if (ni_ttm and cfo_ttm and total_assets) else np.nan

    # ROIC
    invested_cap = (total_equity + total_debt - cash) if (total_equity and total_debt and cash) else np.nan
    nopat = op_ttm * (1 - 0.21) if op_ttm else np.nan  # 법인세율 21% 가정
    feat['roic'] = (nopat / invested_cap * 100) if (nopat and invested_cap) else np.nan

    # delta 피처 (전년 대비 변화)
    roe_prev  = (inc_df['netIncome'].iloc[4:8].sum() / float(bal_df['totalStockholdersEquity'].iloc[4]) * 100) \
                if len(bal_df) > 4 and len(inc_df) >= 8 else np.nan
    feat['delta_roe_1y']    = feat['roe'] - roe_prev if pd.notna(feat['roe']) and pd.notna(roe_prev) else np.nan

    # Piotroski F-Score (간략)
    f = 0
    if pd.notna(feat['roa'])            and feat['roa'] > 0:         f += 1
    if pd.notna(feat['cfo_to_net_income']) and feat['cfo_to_net_income'] > 0: f += 1
    if pd.notna(feat['delta_roe_1y'])   and feat['delta_roe_1y'] > 0: f += 1
    if pd.notna(feat['accruals_ratio']) and feat['accruals_ratio'] < 0: f += 1  # CFO > NI is good
    if pd.notna(feat['debt_to_equity']) and len(bal_df) > 4:
        de_prev = safe_get(bal_df, 'totalDebt', 4) / (safe_get(bal_df, 'totalStockholdersEquity', 4) or 1)
        if feat['debt_to_equity'] < de_prev: f += 1
    if pd.notna(feat['gross_margin']):
        gm_prev = (inc_df['grossProfit'].iloc[4:8].sum() / (inc_df['revenue'].iloc[4:8].sum() or 1) * 100) \
                  if len(inc_df) >= 8 else np.nan
        if pd.notna(gm_prev) and feat['gross_margin'] > gm_prev: f += 1
    if pd.notna(feat['asset_turnover']) and len(bal_df) >= 8:
        at_prev = inc_df['revenue'].iloc[4:8].sum() / (float(bal_df['totalAssets'].iloc[4]) or 1)
        if feat['asset_turnover'] > at_prev: f += 1
    feat['piotroski_f_score'] = f

    # FCF 관련
    shares_out = safe_get(inc_df, 'weightedAverageShsOut')
    feat['fcf_per_share'] = (ttm(cf_df, 'freeCashFlow') / shares_out) if shares_out else np.nan

    return feat

# ── 실행 ──────────────────────────────────────────────────────────────
print('📥 FMP Fundamental 데이터 수집 중...')
fund_features = []
for tk in TEST_TICKERS:
    print(f'  [{tk}]...')
    f = fetch_fmp_financials(tk)
    fund_features.append(f)

df_fund = pd.DataFrame(fund_features).set_index('ticker')
print(f'\n✅ Fundamental 피처 완성: {df_fund.shape[1]}개 피처')
display(df_fund.round(4))

📥 FMP Fundamental 데이터 수집 중...
  [NVDA]...
  [AAPL]...
  [GOOG]...
  [MSFT]...
  [AMZN]...
  [TSM]...
  [AVGO]...
  [META]...
  [TSLA]...
  [WMT]...

✅ Fundamental 피처 완성: 19개 피처


,sales_growth_yoy,eps_growth_yoy,ebitda_growth_yoy,fcf_growth_yoy,roe,roa,gross_margin,operating_margin,debt_to_equity,net_debt_to_ebitda,interest_coverage,asset_turnover,inventory_turnover,cfo_to_net_income,accruals_ratio,roic,delta_roe_1y,piotroski_f_score,fcf_per_share
ticker,,,,,,,,,,,,,,,,,,,
NVDA,65.4735,65.9933,67.8164,58.8681,76.3333,58.0586,71.0681,60.3817,0.0726,0.0056,558.1158,1.0442,2.9190,0.8555,0.0839,65.1523,-15.5395,3,3.9778
AAPL,10.0710,25.5151,11.3198,25.4580,133.5492,31.0514,47.3253,32.3840,1.0263,0.2954,NaN,1.1485,39.0570,1.1502,-0.0467,83.5535,-10.4785,5,8.3620
GOOG,15.1095,34.1943,33.4609,0.6899,31.8279,22.2030,59.6591,32.0441,0.1735,0.2287,NaN,0.6768,NaN,1.2462,-0.0547,22.3382,1.0303,5,6.0686
MSFT,16.6733,28.6287,33.9246,10.5396,30.5115,17.9260,68.5863,46.6713,0.3154,0.5172,72.4120,0.4591,90.6081,1.3458,-0.0620,22.9907,-0.1299,4,10.4174
AMZN,12.3778,29.0265,33.5387,-76.5953,18.8948,9.4946,50.2857,11.1553,0.3722,0.4002,72.7093,0.8764,9.2998,1.7962,-0.0756,13.2386,-1.8234,5,0.7186
TSM,31.9466,48.6239,41.0322,17.3058,31.8756,21.7764,59.8977,50.8077,0.1970,-0.5854,447.8169,0.4828,5.3306,1.3300,-0.0719,41.3294,4.5828,7,196.6072
AVGO,25.2214,145.5814,52.4681,39.4376,31.2650,14.6978,67.0924,40.8673,0.8270,1.3331,12.8446,0.4019,7.5861,1.1887,-0.0277,16.7318,16.8315,7,6.0981
META,22.1678,-2.4807,21.9729,-14.7267,27.8297,16.5176,81.9994,41.4379,0.3862,0.4532,74.8333,0.5491,NaN,1.9154,-0.1512,24.8007,-6.3146,4,18.2900
TSLA,-2.9307,-48.4456,-20.0163,73.6945,4.6191,2.7531,18.0265,4.5926,0.1020,-0.6917,34.8047,0.6881,6.2728,3.8869,-0.0795,4.6493,-5.3530,5,1.9251


---
## Cell 7 — Valuation Features (TTM 기반)
> **소스**: FMP API (key-metrics TTM + 최신 주가)

In [8]:
def fetch_fmp_valuation(ticker: str, price_db: pd.DataFrame, ohlcv_yf: dict, t0: datetime) -> dict:
    """
    FMP key-metrics (TTM) + ratios (TTM)에서 밸류에이션 지표 수집
    가급적 TTM 월별 최신 데이터 반영
    """
    feat = {'ticker': ticker}

    # key-metrics (TTM)
    km  = fmp_get(f'key-metrics/{ticker}',       {'period': 'annual', 'limit': 2})
    km_ttm = fmp_get(f'key-metrics-ttm/{ticker}',  {})
    rat = fmp_get(f'ratios-ttm/{ticker}',         {})

    def g(data, key):
        """리스트 또는 dict에서 키 값 안전 추출"""
        try:
            if isinstance(data, list) and len(data) > 0:
                return float(data[0].get(key, np.nan) or np.nan)
            elif isinstance(data, dict):
                return float(data.get(key, np.nan) or np.nan)
        except:
            return np.nan
        return np.nan

    # TTM 밸류에이션 멀티플
    feat['per']       = g(km_ttm, 'peRatioTTM')    or g(rat, 'priceEarningsRatioTTM')
    feat['pbr']       = g(km_ttm, 'pbRatioTTM')    or g(rat, 'priceToBookRatioTTM')
    feat['psr']       = g(km_ttm, 'priceToSalesRatioTTM') or g(rat, 'priceToSalesRatioTTM')
    feat['ev_ebitda'] = g(km_ttm, 'enterpriseValueOverEBITDATTM') or g(rat, 'enterpriseValueMultipleTTM')
    feat['fcf_yield'] = g(km_ttm, 'freeCashFlowYieldTTM')
    feat['ev_sales']  = g(km_ttm, 'evToSalesTTM')

    # PEG (PER / EPS 성장률)
    # FMP 직접 필드 없으면 계산
    peg = g(rat, 'priceEarningsToGrowthRatioTTM')
    feat['peg_ratio'] = peg

    # Earnings Yield
    ey = (1 / feat['per'] * 100) if feat['per'] and feat['per'] > 0 else np.nan
    feat['earnings_yield'] = ey
    # Earnings Yield Spread (vs 10y yield) — macro_dict 참조
    us10y = macro_dict.get('us10y_yield', np.nan)
    feat['earnings_yield_spread'] = (ey - us10y) if pd.notna(ey) and pd.notna(us10y) else np.nan

    # 3개월 전 밸류에이션 변화율 — historical key-metrics 사용
    km_hist = fmp_get(f'key-metrics/{ticker}', {'period': 'quarter', 'limit': 6})
    def delta_val(km_list, key, n_q=1):
        try:
            cur  = float(km_list[0].get(key) or np.nan)
            prev = float(km_list[n_q].get(key) or np.nan)
            return cur - prev
        except:
            return np.nan

    feat['delta_per_3m']  = delta_val(km_hist, 'peRatio')
    feat['delta_pbr_3m']  = delta_val(km_hist, 'pbRatio')
    feat['delta_psr_3m']  = delta_val(km_hist, 'priceToSalesRatio')

    return feat


# ── 섹터별 Z-Score 계산 (Cell 8 이후에 업데이트) ──────────────────────
def calc_sector_zscore(df: pd.DataFrame, cols: list, sector_map: dict) -> pd.DataFrame:
    df = df.copy()
    df['sector'] = df.index.map(sector_map)
    for col in cols:
        zc = f'{col}_sector_z'
        df[zc] = df.groupby('sector')[col].transform(
            lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0.0
        )
    df = df.drop(columns=['sector'])
    return df


# ── 실행 ──────────────────────────────────────────────────────────────
print('📥 FMP Valuation (TTM) 데이터 수집 중...')
val_features = []
for tk in TEST_TICKERS:
    print(f'  [{tk}]...')
    f = fetch_fmp_valuation(tk, price_db, ohlcv_yf, T0)
    val_features.append(f)

df_val = pd.DataFrame(val_features).set_index('ticker')
print(f'\n✅ Valuation 피처 완성: {df_val.shape[1]}개 피처')
display(df_val.round(4))

📥 FMP Valuation (TTM) 데이터 수집 중...
  [NVDA]...
  [AAPL]...
  [GOOG]...
  [MSFT]...
  [AMZN]...
  [TSM]...
  [AVGO]...
  [META]...
  [TSLA]...
  [WMT]...

✅ Valuation 피처 완성: 12개 피처


,per,pbr,psr,ev_ebitda,fcf_yield,ev_sales,peg_ratio,earnings_yield,earnings_yield_spread,delta_per_3m,delta_pbr_3m,delta_psr_3m
ticker,,,,,,,,,,,,
NVDA,35.9073,27.4093,19.9662,29.8320,0.0224,19.9699,1.6757,2.7849,NaN,-10.1230,-10.3677,-15.1950
AAPL,32.0466,42.7979,8.6349,24.8837,0.0328,8.7386,5.4479,3.1205,NaN,-10.8133,-6.0704,-9.2199
GOOG,27.0170,8.5989,8.8804,20.0293,0.0205,8.9829,4.0645,3.7014,NaN,6.4197,1.5052,4.4705
MSFT,23.2696,7.0999,9.0789,15.0072,0.0279,9.4029,1.6912,4.2974,NaN,-11.3260,-1.4094,-5.3471
AMZN,28.9227,5.4649,3.1410,14.0198,0.0034,3.2333,26.0666,3.4575,NaN,1.5054,-0.3273,-1.4243
TSM,27.3254,8.6615,12.3227,16.7197,0.0200,11.8755,3.3110,3.6596,NaN,1.1642,0.6461,4.2382
AVGO,59.7181,18.6709,21.8413,39.6529,0.0194,22.6011,7.7005,1.6745,NaN,3.0498,-1.4502,-13.9409
META,23.9540,6.6663,7.2062,14.1202,0.0318,7.4452,6.8520,4.1747,NaN,-152.3105,-1.8647,-8.2888
TSLA,307.0813,14.1844,14.2690,114.3278,0.0046,14.1832,-10.9481,0.3256,NaN,171.1445,-0.2551,7.2722


---
## Cell 8 — Sector Features
> **소스**: FMP API (company profile) / FDR (섹터 ETF 수익률)

In [9]:
# GICS 섹터 → ETF 매핑 (SPDR Sector ETF)
SECTOR_ETF_MAP = {
    'Technology':             'XLK',
    'Health Care':            'XLV',
    'Financials':             'XLF',
    'Consumer Discretionary': 'XLY',
    'Consumer Staples':       'XLP',
    'Industrials':            'XLI',
    'Energy':                 'XLE',
    'Utilities':              'XLU',
    'Real Estate':            'XLRE',
    'Materials':              'XLB',
    'Communication Services': 'XLC',
}

def fetch_sector_info(tickers: list) -> pd.DataFrame:
    """FMP profile에서 섹터/산업 정보 수집"""
    rows = []
    for tk in tickers:
        data = fmp_get(f'profile/{tk}', {})
        if data:
            d = data[0]
            rows.append({
                'ticker':   tk,
                'sector':   d.get('sector', 'Unknown'),
                'industry': d.get('industry', 'Unknown'),
                'market_cap_latest': d.get('mktCap', np.nan),
                'country':  d.get('country', 'US'),
            })
        else:
            rows.append({'ticker': tk, 'sector': 'Unknown', 'industry': 'Unknown',
                         'market_cap_latest': np.nan, 'country': 'US'})
    return pd.DataFrame(rows).set_index('ticker')

def fetch_sector_etf_returns(sectors: list, from_date: str) -> dict:
    """SPDR Sector ETF 수익률 수집"""
    sector_rets = {}
    etf_symbols = list(set([SECTOR_ETF_MAP.get(s, 'SPY') for s in sectors]))
    for sym in etf_symbols:
        try:
            df = yf.download(sym, start=from_date, progress=False, auto_adjust=True)
            if not df.empty:
                sector_rets[sym] = df['Close'].squeeze()
        except Exception as e:
            print(f'  ⚠️  ETF [{sym}]: {e}')
    return sector_rets

# ── 실행 ──────────────────────────────────────────────────────────────
print('📥 FMP 섹터 정보 수집 중...')
df_sector_info = fetch_sector_info(TEST_TICKERS)
sector_map = df_sector_info['sector'].to_dict()
print(f'  → 수집 완료')
display(df_sector_info)

print('\n📥 섹터 ETF 수익률 수집 중...')
unique_sectors = df_sector_info['sector'].unique().tolist()
sector_etf_rets = fetch_sector_etf_returns(unique_sectors, MACRO_FROM_LONG)

print('\n⚙️  Sector 피처 계산 중...')
sector_features = []
for tk in TEST_TICKERS:
    sec = sector_map.get(tk, 'Unknown')
    etf = SECTOR_ETF_MAP.get(sec, 'SPY')
    f   = {'ticker': tk, 'sector_id': sec, 'industry_id': df_sector_info.loc[tk, 'industry']}

    # 섹터 ETF 수익률
    if etf in sector_etf_rets:
        s_etf = sector_etf_rets[etf]
        s_etf = s_etf[s_etf.index <= pd.Timestamp(T0)]
        if len(s_etf) > 21:
            f['sector_ret_1m'] = (float(s_etf.iloc[-1]) / float(s_etf.iloc[-21]) - 1) * 100
        if len(s_etf) > 63:
            f['sector_ret_3m'] = (float(s_etf.iloc[-1]) / float(s_etf.iloc[-63]) - 1) * 100
        if len(s_etf) > 60:
            log_r = np.log(s_etf / s_etf.shift(1)).dropna()
            f['sector_vol_60d'] = log_r.iloc[-60:].std() * np.sqrt(252) * 100
    else:
        f['sector_ret_1m'] = f['sector_ret_3m'] = f['sector_vol_60d'] = np.nan

    # 종목 모멘텀 vs 섹터 모멘텀 (stock_vs_sector_momentum)
    stk_ret_1m = df_price.loc[tk, 'ret_1m'] if tk in df_price.index and 'ret_1m' in df_price.columns else np.nan
    f['stock_vs_sector_momentum'] = (stk_ret_1m - f.get('sector_ret_1m', np.nan)) \
        if pd.notna(stk_ret_1m) and pd.notna(f.get('sector_ret_1m')) else np.nan

    sector_features.append(f)

df_sector = pd.DataFrame(sector_features).set_index('ticker')

# 섹터 내 분산도 (dispersion)
for sec in df_sector['sector_id'].unique():
    mask = df_sector['sector_id'] == sec
    tks  = df_sector[mask].index.tolist()
    ret1m_vals = [df_price.loc[t, 'ret_1m'] for t in tks if t in df_price.index and 'ret_1m' in df_price.columns]
    disp = np.std(ret1m_vals) if len(ret1m_vals) > 1 else np.nan
    df_sector.loc[mask, 'sector_dispersion'] = disp

# 섹터 내 Z-Score 업데이트 (Valuation)
val_z_cols = ['per', 'pbr', 'psr', 'ev_ebitda', 'fcf_yield']
df_val = calc_sector_zscore(df_val, [c for c in val_z_cols if c in df_val.columns], sector_map)

print(f'\n✅ Sector 피처 완성: {df_sector.shape[1]}개 피처')
display(df_sector.round(4))

📥 FMP 섹터 정보 수집 중...
  → 수집 완료


,sector,industry,market_cap_latest,country
ticker,,,,
NVDA,Technology,Semiconductors,4311464017763.0005,US
AAPL,Technology,Consumer Electronics,3761492983658.0000,US
GOOG,Communication Services,Internet Content & Information,3562082664168.9995,US
MSFT,Technology,Software - Infrastructure,2773175779800.0000,US
AMZN,Consumer Cyclical,Specialty Retail,2251864350900.0000,US
TSM,Technology,Semiconductors,1758434177829.0000,TW
AVGO,Technology,Semiconductors,1491367493553.0000,US
META,Communication Services,Internet Content & Information,1448209620972.0000,US
TSLA,Consumer Cyclical,Auto - Manufacturers,1353089449111.0000,US



📥 섹터 ETF 수익률 수집 중...

⚙️  Sector 피처 계산 중...

✅ Sector 피처 완성: 7개 피처


,sector_id,industry_id,sector_ret_1m,sector_ret_3m,sector_vol_60d,stock_vs_sector_momentum,sector_dispersion
ticker,,,,,,,
NVDA,Technology,Semiconductors,-5.5219,-2.0576,21.7412,-2.4413,7.2337
AAPL,Technology,Consumer Electronics,-5.5219,-2.0576,21.7412,7.8062,7.2337
GOOG,Communication Services,Internet Content & Information,-1.4525,3.4704,12.3226,-6.5880,2.0835
MSFT,Technology,Software - Infrastructure,-5.5219,-2.0576,21.7412,-3.8806,7.2337
AMZN,Consumer Cyclical,Specialty Retail,-1.1599,1.2266,10.7347,-11.9663,4.8767
TSM,Technology,Semiconductors,-5.5219,-2.0576,21.7412,15.8385,7.2337
AVGO,Technology,Semiconductors,-5.5219,-2.0576,21.7412,2.1415,7.2337
META,Communication Services,Internet Content & Information,-1.4525,3.4704,12.3226,-10.7551,2.0835
TSLA,Consumer Cyclical,Auto - Manufacturers,-1.1599,1.2266,10.7347,-2.2130,4.8767


---
## Cell 9 — Analyst Estimate Revision Features
> **소스**: FMP API (analyst-estimates, price-target, ratings)

In [10]:
def fetch_analyst_features(ticker: str, t0: datetime) -> dict:
    """
    FMP에서 애널리스트 추정치 및 목표주가 수집
    - analyst-estimates (quarterly): EPS/Revenue 컨센서스
    - price-target: 목표주가
    - analyst-stock-recommendations: Buy/Hold/Sell 비율
    """
    feat = {'ticker': ticker}

    # ── EPS / Revenue 컨센서스 추정치 변화율 ─────────────────────────
    est = fmp_get(f'analyst-estimates/{ticker}', {'period': 'quarter', 'limit': 8})
    if est and len(est) >= 2:
        try:
            est_df = pd.DataFrame(est)
            est_df['date'] = pd.to_datetime(est_df['date'])
            est_df = est_df.sort_values('date', ascending=False)

            # FY1 추정: 기준일 이후 첫 번째 미래 분기
            future_est = est_df[est_df['date'] > pd.Timestamp(t0)]
            if not future_est.empty:
                fy1 = future_est.iloc[0]
                # EPS 컨센서스 변화율 (현재 추정 / 직전 분기 추정 비교는 FMP에서 직접 제공 안 됨 → 근사)
                eps_cur  = float(fy1.get('estimatedEpsAvg')  or np.nan)
                eps_low  = float(fy1.get('estimatedEpsLow')  or np.nan)
                eps_high = float(fy1.get('estimatedEpsHigh') or np.nan)
                rev_cur  = float(fy1.get('estimatedRevenueAvg') or np.nan)
                # Revision Breadth: (High - Low) / Avg — 폭이 좁을수록 컨센서스 수렴
                feat['eps_consensus_fy1']      = eps_cur
                feat['revenue_consensus_fy1']  = rev_cur
                feat['eps_revision_breadth']   = ((eps_high - eps_low) / abs(eps_cur)) * 100 \
                                                  if eps_cur and eps_cur != 0 else np.nan
        except Exception as e:
            print(f'  ⚠️  {ticker} analyst-estimates 파싱 오류: {e}')

    # ── 목표주가 upside ───────────────────────────────────────────────
    pt = fmp_get(f'price-target/{ticker}', {'limit': 20})
    if pt:
        try:
            pt_df = pd.DataFrame(pt)
            pt_df['publishedDate'] = pd.to_datetime(pt_df['publishedDate'])
            recent_pts = pt_df[pt_df['publishedDate'] >= pd.Timestamp(t0) - timedelta(days=90)]
            if not recent_pts.empty:
                avg_target = recent_pts['priceTarget'].astype(float).mean()
                analyst_count = recent_pts['analystName'].nunique() if 'analystName' in recent_pts.columns else len(recent_pts)
                feat['analyst_count_3m'] = analyst_count
                # 현재 주가 가져오기
                cur_price = np.nan
                if ticker in price_db.columns:
                    s = price_db[ticker].dropna()
                    s = s[s.index <= pd.Timestamp(t0)]
                    cur_price = float(s.iloc[-1]) if len(s) > 0 else np.nan
                elif ticker in ohlcv_yf:
                    yf_s = ohlcv_yf[ticker]['close']
                    yf_s = yf_s[yf_s.index <= pd.Timestamp(t0)]
                    cur_price = float(yf_s.iloc[-1]) if len(yf_s) > 0 else np.nan
                if pd.notna(cur_price) and cur_price > 0:
                    feat['target_price_upside'] = (avg_target / cur_price - 1) * 100
                # 1개월 내 vs 3개월 내 목표주가 변화
                pt_1m = pt_df[pt_df['publishedDate'] >= pd.Timestamp(t0) - timedelta(days=30)]
                pt_3m = pt_df[pt_df['publishedDate'] >= pd.Timestamp(t0) - timedelta(days=90)]
                if not pt_1m.empty and not pt_3m.empty:
                    avg_1m = pt_1m['priceTarget'].astype(float).mean()
                    avg_3m = pt_3m['priceTarget'].astype(float).mean()
                    feat['target_price_revision_1m'] = (avg_1m / avg_3m - 1) * 100 if avg_3m != 0 else np.nan
        except Exception as e:
            print(f'  ⚠️  {ticker} price-target 파싱 오류: {e}')

    # ── Buy/Hold/Sell 비율 ────────────────────────────────────────────
    ratings = fmp_get(f'analyst-stock-recommendations/{ticker}', {'limit': 1})
    if ratings:
        try:
            r = ratings[0]
            buy  = float(r.get('analystRatingsbuy', 0)  or 0)
            hold = float(r.get('analystRatingsHold', 0) or 0)
            sell = float(r.get('analystRatingsSell', 0) or 0)
            total = buy + hold + sell
            feat['buy_ratio']  = buy  / total * 100 if total > 0 else np.nan
            feat['hold_ratio'] = hold / total * 100 if total > 0 else np.nan
            feat['sell_ratio'] = sell / total * 100 if total > 0 else np.nan
        except Exception as e:
            print(f'  ⚠️  {ticker} ratings 파싱 오류: {e}')

    # ── Earnings Surprise (최근 분기) ─────────────────────────────────
    surp = fmp_get(f'earnings-surprises/{ticker}', {})
    if surp:
        try:
            sp_df = pd.DataFrame(surp)
            sp_df['date'] = pd.to_datetime(sp_df['date'])
            sp_df = sp_df.sort_values('date', ascending=False)
            past = sp_df[sp_df['date'] <= pd.Timestamp(t0)]
            if len(past) >= 1:
                act   = float(past['actualEarningResult'].iloc[0] or np.nan)
                est_v = float(past['estimatedEarning'].iloc[0]     or np.nan)
                feat['earnings_surprise_1q'] = ((act - est_v) / abs(est_v) * 100) \
                                                if est_v and est_v != 0 else np.nan
            if len(past) >= 3:
                surprises = []
                for i in range(min(3, len(past))):
                    a = float(past['actualEarningResult'].iloc[i] or np.nan)
                    e = float(past['estimatedEarning'].iloc[i]    or np.nan)
                    if e and e != 0:
                        surprises.append((a - e) / abs(e) * 100)
                feat['earnings_surprise_3q_avg'] = np.mean(surprises) if surprises else np.nan
        except Exception as e:
            print(f'  ⚠️  {ticker} earnings-surprises 파싱 오류: {e}')

    return feat

# ── 실행 ──────────────────────────────────────────────────────────────
print('📥 FMP Analyst Estimate 데이터 수집 중...')
analyst_features = []
for tk in TEST_TICKERS:
    print(f'  [{tk}]...')
    f = fetch_analyst_features(tk, T0)
    analyst_features.append(f)

df_analyst = pd.DataFrame(analyst_features).set_index('ticker')
print(f'\n✅ Analyst Revision 피처 완성: {df_analyst.shape[1]}개 피처')
display(df_analyst.round(4))

📥 FMP Analyst Estimate 데이터 수집 중...
  [NVDA]...
  [AAPL]...
  [GOOG]...
  [MSFT]...
  [AMZN]...
  [TSM]...
  [AVGO]...
  [META]...
  [TSLA]...
  [WMT]...

✅ Analyst Revision 피처 완성: 8개 피처


,eps_consensus_fy1,revenue_consensus_fy1,eps_revision_breadth,buy_ratio,hold_ratio,sell_ratio,earnings_surprise_1q,earnings_surprise_3q_avg
ticker,,,,,,,,
NVDA,3.4433,149137802835.0000,19.6036,94.1176,3.9216,1.9608,5.1948,4.1099
AAPL,2.4048,123401826658.0000,14.4344,60.9756,36.5854,2.4390,6.3670,7.4437
GOOG,4.2756,172734925649.0000,10.0716,89.0909,10.9091,0.0000,7.2243,12.6567
MSFT,5.8895,115153773756.0000,6.1406,93.7500,6.2500,0.0000,6.1538,8.9988
AMZN,5.8896,297957450310.0000,7.6387,92.4528,7.5472,0.0000,-1.0152,17.1443
TSM,159.5297,1719209413478.0000,13.0775,92.3077,7.6923,0.0000,6.5517,8.1935
AVGO,5.9965,51624185674.0000,36.7731,95.2381,4.7619,0.0000,4.2781,2.2407
META,16.7144,127398184000.0000,11.9419,89.2857,10.7143,0.0000,8.4249,12.5801
TSLA,NaN,50307830000.0000,NaN,43.9024,41.4634,14.6341,9.9384,0.0830


---
## Cell 10 — 피처 통합 및 최종 결과 확인

In [13]:
print('⚙️  전체 피처 통합 중...')

# ── 1. 종목별 피처 (wide 형태로 join) ──────────────────────────────
df_all = df_price.copy()

for df_part, name in [
    (df_fund,     'Fundamental'),
    (df_val,      'Valuation'),
    (df_sector,   'Sector'),
    (df_analyst,  'Analyst'),
]:
    before = df_all.shape[1]
    df_all = df_all.join(df_part, how='left', rsuffix=f'_{name[:3].lower()}')
    print(f'  + {name}: {df_all.shape[1] - before}개 피처 추가 → 누계 {df_all.shape[1]}개')

# ── 2. Macro / Commodity — 전 종목 공통값 broadcast ──────────────
for k, v in macro_dict.items():
    df_all[k] = v
for k, v in comm_dict.items():
    df_all[k] = v
print(f'  + Macro: {len(macro_dict)}개 피처 broadcast')
print(f'  + Commodity: {len(comm_dict)}개 피처 broadcast')

# ── 3. 기준일 컬럼 추가 ────────────────────────────────────────────
df_all.insert(0, 'feature_date', T0_STR)

# print(f'\n{'='*60}')
print(f'✅ 최종 피처 매트릭스')
print(f'   종목 수  : {len(df_all)}개')
print(f'   피처 수  : {df_all.shape[1]}개')
print(f'   기준일   : {T0_STR}')
null_pct = df_all.isnull().mean().mean() * 100
print(f'   결측율   : {null_pct:.1f}%')
# print(f'{'='*60}\n')

display(df_all.round(4))

⚙️  전체 피처 통합 중...
  + Fundamental: 19개 피처 추가 → 누계 41개
  + Valuation: 17개 피처 추가 → 누계 58개
  + Sector: 7개 피처 추가 → 누계 65개
  + Analyst: 8개 피처 추가 → 누계 73개
  + Macro: 15개 피처 broadcast
  + Commodity: 11개 피처 broadcast
✅ 최종 피처 매트릭스
   종목 수  : 10개
   피처 수  : 99개
   기준일   : 2026-03-01
   결측율   : 10.8%


,feature_date,ret_1m,ret_3m,ret_6m,ret_12m,ret_12m_ex_1m,price_to_ma20,price_to_ma60,price_to_ma120,ma20_to_ma60,ma60_to_ma120,vol_20d,vol_60d,downside_vol_60d,vol_change_20d_60d,drawdown_252d,price_to_high_252d,price_to_low_252d,volume_zscore_20d,volume_change_1m,dollar_volume_60d,amihud_illiquidity,rsi_14d,sales_growth_yoy,eps_growth_yoy,ebitda_growth_yoy,fcf_growth_yoy,roe,roa,gross_margin,operating_margin,debt_to_equity,net_debt_to_ebitda,interest_coverage,asset_turnover,inventory_turnover,cfo_to_net_income,accruals_ratio,roic,delta_roe_1y,piotroski_f_score,fcf_per_share,per,pbr,psr,ev_ebitda,fcf_yield,ev_sales,peg_ratio,earnings_yield,earnings_yield_spread,delta_per_3m,delta_pbr_3m,delta_psr_3m,per_sector_z,pbr_sector_z,psr_sector_z,ev_ebitda_sector_z,fcf_yield_sector_z,sector_id,industry_id,sector_ret_1m,sector_ret_3m,sector_vol_60d,stock_vs_sector_momentum,sector_dispersion,eps_consensus_fy1,revenue_consensus_fy1,eps_revision_breadth,buy_ratio,hold_ratio,sell_ratio,earnings_surprise_1q,earnings_surprise_3q_avg,us10y_yield,us2y_yield,term_spread_10y_2y,term_spread_change_1m,real_rate_proxy,vix,vix_change_1m,oas_spread,credit_spread_change_1m,dxy,dxy_change_1m,pmi,pmi_change_1m,unemployment_rate,wti_ret_1m,wti_ret_3m,copper_ret_1m,copper_ret_3m,gold_ret_1m,gold_ret_3m,natgas_ret_1m,natgas_ret_3m,corn_ret_1m,corn_ret_3m,wti_curve_structure
ticker,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NVDA,2026-03-01,-7.9632,-1.7032,-1.6486,47.5025,60.2647,-4.7278,-4.1770,-4.1646,0.5781,0.0130,45.7928,34.8305,24.0967,31.4732,-14.4182,-14.4182,87.9096,2.0214,24.9463,32218073144.3352,0.0000,41.0962,65.4735,65.9933,67.8164,58.8681,76.3333,58.0586,71.0681,60.3817,0.0726,0.0056,558.1158,1.0442,2.9190,0.8555,0.0839,65.1523,-15.5395,3,3.9778,35.9073,27.4093,19.9662,29.8320,0.0224,19.9699,1.6757,2.7849,NaN,-10.1230,-10.3677,-15.1950,0.0178,0.4402,0.9074,0.4577,-0.3611,Technology,Semiconductors,-5.5219,-2.0576,21.7412,-2.4413,7.2337,3.4433,149137802835.0000,19.6036,94.1176,3.9216,1.9608,5.1948,4.1099,NaN,NaN,0.0000,NaN,NaN,19.2800,1.2100,NaN,NaN,97.6100,-0.1800,NaN,NaN,NaN,2.9134,12.5754,-3.2077,18.7033,-4.2284,25.6285,-24.3185,-36.8280,0.6395,2.3050,-11.8487
AAPL,2026-03-01,2.2843,-4.8172,13.7089,11.7040,9.2093,-1.6520,-1.3118,0.3840,0.3459,1.7183,34.0466,23.4063,17.7798,45.4591,-7.6907,-7.6907,53.7360,0.9411,20.4672,12925113926.2728,0.0000,38.9249,10.0710,25.5151,11.3198,25.4580,133.5492,31.0514,47.3253,32.3840,1.0263,0.2954,NaN,1.1485,39.0570,1.1502,-0.0467,83.5535,-10.4785,5,8.3620,32.0466,42.7979,8.6349,24.8837,0.0328,8.7386,5.4479,3.1205,NaN,-10.8133,-6.0704,-9.2199,-0.2527,1.4854,-0.9295,-0.0333,1.4467,Technology,Consumer Electronics,-5.5219,-2.0576,21.7412,7.8062,7.2337,2.4048,123401826658.0000,14.4344,60.9756,36.5854,2.4390,6.3670,7.4437,NaN,NaN,0.0000,NaN,NaN,19.2800,1.2100,NaN,NaN,97.6100,-0.1800,NaN,NaN,NaN,2.9134,12.5754,-3.2077,18.7033,-4.2284,25.6285,-24.3185,-36.8280,0.6395,2.3050,-11.8487
GOOG,2026-03-01,-8.0405,-2.6994,46.8663,83.6803,99.7405,-2.0828,-2.6260,6.4053,-0.5547,9.2748,24.4446,22.0610,14.3087,10.8043,-9.7043,-9.7043,113.0456,0.8656,32.4377,7046832346.1526,0.0000,38.7723,15.1095,34.1943,33.4609,0.6899,31.8279,22.2030,59.6591,32.0441,0.1735,0.2287,NaN,0.6768,NaN,1.2462,-0.0547,22.3382,1.0303,5,6.0686,27.0170,8.5989,8.8804,20.0293,0.0205,8.9829,4.0645,3.7014,NaN,6.4197,1.5052,4.4705,0.7071,0.7071,0.7071,0.7071,-0.7071,Communication Services,Internet Content & Information,-1.4525,3.4704,12.3226,-6.5880,2.0835,4.2756,172734925649.0000,10.0716,89.0909,10.9091,0.0000,7.2243,12.6567,NaN,NaN,0.0000,NaN,NaN,19.2800,1.2100,NaN,NaN,97.6100,-0.1800,NaN,NaN,NaN,2.9134,12.5754,-3.2077,18.7033,-4.2284,25.6285,-24.3185,-36.8280,0.6395,2.3050,-11.8487
MSFT,2026-03-01,-9.4025,-19.1061,-22.7938,0.5891,11.0286,-2.5079,-13.0007,-18.1821,-10.7627,-5.9556,32.4255,32.9748,32.1187,-1.6660,-27.4129,-27.4129,11.3619,0.7545,83.8980,14366159918.8136,0.0000,44.8238,16.6733,28.6287,33.9246,10

---
## Cell 11 — 피처별 결측 현황 및 카테고리별 요약

In [14]:
# ── 결측 현황 요약 ────────────────────────────────────────────────────
null_summary = df_all.isnull().sum().reset_index()
null_summary.columns = ['feature', 'null_count']
null_summary['null_pct'] = (null_summary['null_count'] / len(df_all) * 100).round(1)
null_summary = null_summary[null_summary['null_count'] > 0].sort_values('null_pct', ascending=False)

if not null_summary.empty:
    print(f'⚠️  결측값 있는 피처 ({len(null_summary)}개):')
    display(null_summary)
else:
    print('✅ 결측값 없음')

# ── 카테고리별 피처 수 요약 ────────────────────────────────────────────
category_map = {
    'Price/Technical': ['ret_','vol_','price_to_ma','ma2','ma6','drawdown','amihud','rsi',
                        'volume_','dollar_','beta_','idio','rel_ret'],
    'Macro':           ['us10y','us2y','term_spread','vix','oas','dxy','pmi','unemployment',
                        'real_rate','fed_','global_'],
    'Commodity':       ['wti','copper','gold','natgas','corn'],
    'Fundamental':     ['sales_growth','eps_growth','ebitda_growth','roe','roa','margin',
                        'debt_to','net_debt','interest_cov','turnover','delta_roe','delta_margin',
                        'accruals','cfo_to','roic','piotroski','fcf_growth','fcf_per'],
    'Valuation':       ['per','pbr','psr','ev_','fcf_yield','peg','earnings_yield','delta_per',
                        'delta_pbr','delta_psr'],
    'Sector':          ['sector_'],
    'Analyst':         ['eps_consensus','revenue_consensus','eps_revision','target_price',
                        'buy_ratio','hold_ratio','sell_ratio','earnings_surprise','analyst_count'],
}

print('\n📋 카테고리별 피처 수 요약:')
summary_rows = []
for cat, prefixes in category_map.items():
    matched = [c for c in df_all.columns
               if any(c.lower().startswith(p.lower()) or p.lower() in c.lower() for p in prefixes)]
    matched = list(set(matched))
    summary_rows.append({'카테고리': cat, '피처 수': len(matched), '피처 목록': ', '.join(sorted(matched)[:8]) + ('...' if len(matched) > 8 else '')})
    
df_summary = pd.DataFrame(summary_rows)
display(df_summary)
print(f'\n🎯 총 피처 수: {df_all.shape[1] - 1}개 (feature_date 제외)')

# ── 저장 옵션 안내 ─────────────────────────────────────────────────────
print('\n' + '='*60)
print('💾 저장 옵션 (필요 시 활성화):')
print('  CSV 저장: df_all.to_csv("features_test.csv")')
print('  Pickle : df_all.to_pickle("features_test.pkl")')
print('='*60)

⚠️  결측값 있는 피처 (15개):


,feature,null_count,null_pct
50,earnings_yield_spread,10,100.0000
74,us10y_yield,10,100.0000
75,us2y_yield,10,100.0000
77,term_spread_change_1m,10,100.0000
78,real_rate_proxy,10,100.0000
81,oas_spread,10,100.0000
82,credit_spread_change_1m,10,100.0000
85,pmi,10,100.0000
86,pmi_change_1m,10,100.0000
87,unemployment_rate,10,100.0000



📋 카테고리별 피처 수 요약:


,카테고리,피처 수,피처 목록
0,Price/Technical,34,"amihud_illiquidity, copper_ret_1m, copper_ret_..."
1,Macro,13,"dxy, dxy_change_1m, oas_spread, pmi, pmi_chang..."
2,Commodity,11,"copper_ret_1m, copper_ret_3m, corn_ret_1m, cor..."
3,Fundamental,19,"accruals_ratio, asset_turnover, cfo_to_net_inc..."
4,Valuation,22,"copper_ret_1m, copper_ret_3m, delta_pbr_3m, de..."
5,Sector,11,"ev_ebitda_sector_z, fcf_yield_sector_z, pbr_se..."
6,Analyst,8,"buy_ratio, earnings_surprise_1q, earnings_surp..."



🎯 총 피처 수: 98개 (feature_date 제외)

💾 저장 옵션 (필요 시 활성화):
  CSV 저장: df_all.to_csv("features_test.csv")
  Pickle : df_all.to_pickle("features_test.pkl")
